# D2-07 Case study - Regionalised hydrogen pathways

This is the integrative Day 2 exercise. We compare three electrolytic hydrogen production routes across regions and interpret how regionalized LCIA changes the story.


## Learning goals

After this notebook, you should be able to:

- inspect the bundled hydrogen foreground workbook and identify the main technologies
- prepare a comparison table across technologies, regions, and indicators
- connect a foreground hydrogen model to regional electricity backgrounds
- structure and interpret a regionalized comparison for water scarcity and biodiversity


## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Seitfudem, G., Berger, M., Schmied, H. M., & Boulay, A. (2025). The updated and improved method for water scarcity impact assessment in LCA, AWARE2.0. *Journal of Industrial Ecology, 29*(3), 891-907. https://doi.org/10.1111/jiec.70023
- Scherer, L., Rosa, F., Sun, Z., Michelsen, O., De Laurentiis, V., Marques, A., Pfister, S., Verones, F., & Kuipers, K. J. J. (2023). Biodiversity Impact Assessment Considering Land Use Intensities and Fragmentation. *Environmental Science & Technology, 57*(48), 19612-19623. https://doi.org/10.1021/acs.est.3c04191


## 1) Inspect the bundled hydrogen workbook

The Day 2 asset already contains several electrolyser technologies in an importable Excel workbook.


In [ ]:
from pathlib import Path
import openpyxl
import numpy as np
import pandas as pd

foreground_path = Path('tutorials/DAY 2/assets/lci_hydrogen_electrolysis_v3_10_importable.xlsx')
if not foreground_path.exists():
    foreground_path = Path('assets/lci_hydrogen_electrolysis_v3_10_importable.xlsx')

print('Foreground path:', foreground_path.resolve())

wb = openpyxl.load_workbook(foreground_path, read_only=True, data_only=True)
ws = wb['lci']

activity_names = [
    row[1]
    for row in ws.iter_rows(values_only=True)
    if row and row[0] == 'Activity' and row[1]
]

pd.DataFrame({'activity': activity_names})


## 2) Identify the three main hydrogen pathways

The core comparison for this notebook is:

- alkaline electrolysis (AEC)
- proton exchange membrane electrolysis (PEM)
- solid oxide electrolysis (SOEC)


In [ ]:
technology_table = pd.DataFrame([
    {'technology': 'AEC', 'activity contains': 'from AEC electrolysis'},
    {'technology': 'PEM', 'activity contains': 'from PEM electrolysis'},
    {'technology': 'SOEC', 'activity contains': 'from SOEC electrolysis, from grid electricity'},
])
technology_table


## 3) Search candidate regional electricity backgrounds in the BAFU database

The questionnaire asks us to compare regions rather than use the old electricity-mix scenario table. We therefore start by locating country-level electricity activities, for example in Spain and France.


In [ ]:
import bw2data as bd

db_name = next((name for name in sorted(bd.databases) if name.startswith('bafu')), None)

if db_name is None:
    print('No BAFU database found in the current project.')
else:
    db = bd.Database(db_name)
    electricity_hits = [
        act for act in db
        if 'electricity' in act['name'].lower()
        and 'low voltage' in act['name'].lower()
        and act.get('location') in {'ES', 'FR', 'CH'}
    ]
    preview = pd.DataFrame([
        {'name': act['name'], 'location': act.get('location'), 'unit': act.get('unit')}
        for act in electricity_hits[:20]
    ])
    display(preview)


## 4) Optional import scaffold for the hydrogen foreground

Use this section if you want to import the hydrogen workbook into the current project. The code is split into small steps so that you can debug each stage separately.


In [ ]:
import bw2io as bi

importer = bi.ExcelImporter(foreground_path)
print('Unlinked objects before strategies:', len(importer.unlinked))


In [ ]:
importer.apply_strategies()
importer.statistics()


In [ ]:
# If the current project already contains a BAFU database and a biosphere database,
# match them before writing the foreground.
if db_name is not None:
    importer.match_database(db_name)
biosphere_candidates = [name for name in bd.databases if 'biosphere' in name]
if biosphere_candidates:
    importer.match_database(biosphere_candidates[0], kind='biosphere')
importer.statistics()


## Checkpoint 1

Build the comparison plan you want to execute today.

At minimum, define:

- three hydrogen technologies
- two regional electricity backgrounds
- two regionalized indicators
- one hypothesis about how the ranking may change


In [ ]:
# TODO
experiment_plan = pd.DataFrame([
    {'technology': '', 'region': '', 'indicator': '', 'hypothesis': ''},
])
experiment_plan


In [ ]:
experiment_plan = pd.DataFrame([
    {'technology': 'AEC', 'region': 'ES', 'indicator': 'AWARE-like water scarcity', 'hypothesis': 'Spain may score higher if the electricity background is linked to more stressed water withdrawals.'},
    {'technology': 'PEM', 'region': 'ES', 'indicator': 'AWARE-like water scarcity', 'hypothesis': 'PEM may be sensitive to electricity demand and water context.'},
    {'technology': 'SOEC', 'region': 'FR', 'indicator': 'Biodiversity-like', 'hypothesis': 'SOEC may shift rank if the electricity background and land-related factors differ strongly by region.'},
])
experiment_plan


## 5) Create a result table template

This table keeps the case study structured even if different teams make slightly different modeling choices.


In [ ]:
results_template = pd.DataFrame([
    {'technology': 'AEC', 'region': 'ES', 'indicator': 'water scarcity', 'score': np.nan},
    {'technology': 'AEC', 'region': 'FR', 'indicator': 'water scarcity', 'score': np.nan},
    {'technology': 'PEM', 'region': 'ES', 'indicator': 'water scarcity', 'score': np.nan},
    {'technology': 'PEM', 'region': 'FR', 'indicator': 'water scarcity', 'score': np.nan},
    {'technology': 'SOEC', 'region': 'ES', 'indicator': 'water scarcity', 'score': np.nan},
    {'technology': 'SOEC', 'region': 'FR', 'indicator': 'water scarcity', 'score': np.nan},
    {'technology': 'AEC', 'region': 'ES', 'indicator': 'biodiversity', 'score': np.nan},
    {'technology': 'AEC', 'region': 'FR', 'indicator': 'biodiversity', 'score': np.nan},
    {'technology': 'PEM', 'region': 'ES', 'indicator': 'biodiversity', 'score': np.nan},
    {'technology': 'PEM', 'region': 'FR', 'indicator': 'biodiversity', 'score': np.nan},
    {'technology': 'SOEC', 'region': 'ES', 'indicator': 'biodiversity', 'score': np.nan},
    {'technology': 'SOEC', 'region': 'FR', 'indicator': 'biodiversity', 'score': np.nan},
])
results_template


## 6) Interpretation questions for the live exercise

Use your result table to answer:

- Does the technology ranking stay stable across regions?
- Does the ranking stay stable across impact categories?
- Where do you see a possible trade-off between water scarcity and biodiversity?
- Which background electricity choices matter most for the conclusion?


## Recap

After this notebook, you should now be able to:

- inspect and prepare the bundled hydrogen foreground asset
- frame a regionalized comparison across technologies and regions
- organize a Day 2 case study around technologies, regions, and indicators
- interpret how regionalization can change technology rankings and trade-offs
